In [ ]:
import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
import geopandas as gpd
import seaborn as sns
import networkx as nx
from itertools import combinations

# Plot (include no regime, without 2001 and 2002)

In [ ]:
wr = plt.figure(figsize=(15, 15), constrained_layout=True)

gs_out = wr.add_gridspec(2, 1, height_ratios=(1, 1), hspace = 0.05) 
upper = wr.add_subfigure(gs_out[0, 0])  
lower = wr.add_subfigure(gs_out[1, 0])  
gs_in = lower.add_gridspec(3, 2, width_ratios=(1, 1), height_ratios = (1, 1, 1))
rel_conpro = lower.add_subfigure(gs_in[:, 1])     
contri = lower.add_subfigure(gs_in[0:2, 0])  

#get axes
ax_upper = upper.subplots(1, 3).flatten()  
ax_rel_conpro = rel_conpro.subplots(3, 1).flatten()
ax_contri = contri.subplots(1, 1) 

#################################################################################
#upper part
season_dict = dict(zip([0, 1, 2], ["MAM", "JJA", "SON"]))
title_dict = dict(zip([0, 1, 2], ["(a) MAM", "(b) JJA", "(c) SON"]))

for ax_id, ax in enumerate(ax_upper):


    #------------------------------------
    study_area = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/Study_Area/Study_Area_32E.shp")    
    bbox_adjusted = gpd.read_file("/net/rain/hyclimm/data/projects/SynFire/WP1/Regionalization/Region_Ten.shp")   
    
    region_positions = {}
    region = ["BI", "IP", "FR", "ME", "AL", "SEA", "NEA", "SC", "WMD", "EMD"]  
    
    region_positions['BI'] = (-5, 57)
    region_positions['IP'] = (-5, 39.0)
    region_positions['FR'] = (-2, 47)
    region_positions['ME'] = (11.5, 53)
    region_positions['AL'] = (11, 45.5)
    region_positions['SEA'] = (23, 47)
    region_positions['NEA'] = (28, 57)
    region_positions['SC'] = (17, 68)
    region_positions['WMD'] = (12, 36)
    region_positions['EMD'] = (28, 39)
    
    #------------------------------------
    season = season_dict[ax_id]
    
    #load dependency
    dependency = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair_wo_2001_2002/Dependency_wr_for_region_pair_{season}_incl_noregime_wo_2001_2002.csv")
    
    #load ensemble dependency
    dependency_ens = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Analyze_synchronized_event_pair_wo_2001_2002/Dependency_ens2000_wr_for_region_pair_{season}_incl_noregime_wo_2001_2002.csv")
    
    #construct network
    #--------------------------------
    G = nx.MultiGraph()
    
    #add nodes
    G.add_nodes_from(region)

    #small changes to region abbreviations
    node_labels = {
    'BI':'BI',
    'IP':'IP',
    'FR':'FR',
    'ME':'CE', #ME->CE
    'AL':'AL',
    'SEA':'SEE',  #SEA->SEE
    'NEA':'NEE', #NEA->NEE
    'SC':'SC',
    'WMD':'WMD',
    'EMD':'EMD'
    }
    nx.set_node_attributes(G, node_labels, "label")
    
    #weather regime color scheme
    wr_color_dict = dict({
            
            "AR": "#ffd700", #255,215,0
            "AT": "#4b0082",  #75,0,130
            "EuBL": "#44aa99", #68,170,153
            "GL": "#0000ff", #0,0,255
            "ScBL": "#117733", #17,119,51
            "ScTr": "#ddcc77", #221,204,119
            "ZO": "#cc6677", #204,102,119
            "no": "#808080" #128,128,128.
        
        })
    
    
    #add edges
    for (reg1, reg2) in [(combo[0], combo[1]) for combo in combinations(region, 2)]:
    
        #check observation numbers
        n_obs = dependency.loc[(dependency["reg1"] == reg1) & (dependency["reg2"] == reg2), "n"].item()
    
        if n_obs <= 20:
            continue
            
        else:
            dependency_ens_reg = dependency_ens[(dependency_ens["reg1"] == reg1) & (dependency_ens["reg2"] == reg2)]
    
            n = 0
            rad_dict = dict(zip([0, 1, 2], [0, -0.1, 0.1]))   #curvature for network visualization
            
            for wrname in ["EuBL", "GL", "ScBL", "ScTr", "AR", "AT", "ZO", "no"]:
                
                dp = dependency.loc[(dependency["reg1"] == reg1) & (dependency["reg2"] == reg2), wrname].item()
                sig_thd = dependency_ens_reg[wrname].quantile(0.95)
    
                if dp > sig_thd:
                    
                    #add edge
                    scaling_factor = 3   #adjust edge width here
                    G.add_edge(reg1, reg2, rad = rad_dict[n], color = wr_color_dict[wrname], weight = 1 * scaling_factor)   #equal linewidth
                    n += 1 
    
    #--------------------------------------------------------------------------------------
    #Draw the study area and regionalization
    study_area.boundary.plot(ax = ax, color = '#dddddd', zorder = 0)
    bbox_adjusted.boundary.plot(ax = ax, color='#aaaaaa', zorder = 0)
    
    # Draw edges
    for u, v, key, data in G.edges(keys=True, data=True):
        edge_color = data.get('color')  
        edge_weight = data.get('weight')  
        rad = data.get('rad')  
    
        # Draw the individual edge with attributes
        nx.draw_networkx_edges(
            G,
            pos = region_positions,
            ax = ax,
            edgelist=[(u, v)],  # Plot a single edge at a time
            edge_color=[edge_color],
            style = '-', 
            width=edge_weight,
            connectionstyle=f'arc3,rad={rad}'  # Add curvature for multi-edges
        )
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos = region_positions, ax=ax, node_color="#555555", node_size=1100).set_zorder(1)
    
    # Draw labels
    nx.draw_networkx_labels(G, pos = region_positions, ax=ax, labels = node_labels, font_color = "white", font_size=13, font_weight = "bold")
    
    
    # Subpanel title
    ax.text(0.02, 0.95, title_dict[ax_id], fontsize = 18, ha = 'left', va = 'bottom', transform = ax.transAxes)
    
    # turn the axis on
    ax.set_axis_on()
    ax.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)
    ax.set_xlabel("")
    ax.set_ylabel("")


#title
upper.suptitle("Pair-wise synchronicity", fontsize = 20, x = 0, y = 0.99, ha = 'left', va = 'bottom')

#################################################################################
#dependency

for dfkey, ax in zip(["MAM", "JJA", "SON"], ax_rel_conpro):

    #load dependency table, leave out reginal synchronicity values, leave out no regimes
    df = pd.read_csv(f"/net/rain/hyclimm/data/projects/SynFire/WP1/Weather_Regime_Dependency_wo_2001_2002/Weather_Regime_Dependency_five_levels_{dfkey}_incl_noregime_wo_2001_2002.csv")
    sample_size = list(df["n"])


    #melt
    df_melt = df.melt(id_vars = 'synlevel',
                      value_vars = ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL", "no"],
                      var_name = "WR",
                      value_name = "dependency")

    ax.axvline(x = 1.0, color = "#909090", linestyle = "--")
    sns.scatterplot(x='dependency', y='synlevel', hue='WR', data=df_melt, palette=wr_color_dict, marker="x", lw = 2, s = 80, ax = ax)
    title = "(e) MAM" if dfkey == "MAM" else "(f) JJA" if dfkey == "JJA" else "(g) SON"
    ax.text(0.83, 0.77, title, va = 'bottom', ha = 'left', fontsize = 18, transform = ax.transAxes)

    ax.set_ylabel('')
    ax.set_yticks([0, 1, 2, 3, 4])
    ax.set_yticklabels([f'No\nn={sample_size[0]}', f'Regional\nn={sample_size[1]}', f'Low\nn={sample_size[2]}', f'Medium\nn={sample_size[3]}', f'High\nn={sample_size[4]}'], fontsize=12)
    ax.set_xlim(-0.1, 3.9)
    ax.legend().set_visible(False)
    
    if dfkey == "SON":
        ax.set_xlabel("Relative conditional probability (-)", fontsize = 15)
        ax.set_xticks([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5])
        ax.set_xticklabels([0, 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5], fontsize = 13)
    else:
        ax.set_xlabel("")
        ax.set_xticks([])
        ax.set_xticklabels([])

    
    #add lines
    for wrt in ["AT", "ZO", "ScTr", "AR", "EuBL", "ScBL", "GL"]:

        if df[wrt].is_monotonic_increasing or df[wrt].is_monotonic_decreasing:
            val = list(df[wrt])
            ax.plot([val[0], val[1]], [0, 1], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[1], val[2]], [1, 2], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[2], val[3]], [2, 3], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)
            ax.plot([val[3], val[4]], [3, 4], color = wr_color_dict[wrt], linestyle = '-', linewidth = 1)

rel_conpro.suptitle("Continental synchronicity", fontsize = 20, x = 0, y = 1, ha = 'left', va = 'bottom')

#################################################################################
#contribution
MAM_regimes = ["EuBL", "ZO", "ScBL", "ScTr"]
MAM_count = [8, 2, 3, 1] #14
JJA_regimes = ["AR", "ScBL", "AT", "EuBL", "ZO"]
JJA_count = [4, 2, 3, 1, 2] #12
SON_regimes = ["ZO", "ScTr", "EuBL"]
SON_count = [3, 1, 1] #5

regimes = ["EuBL", "ZO", "ScBL", "AR", "AT", "ScTr", "no", "GL"]
frac = [10/31, 7/31, 5/31, 4/31, 3/31, 2/31, 0, 0]

ax_contri.bar(regimes, frac, color = [wr_color_dict[regime] for regime in regimes], edgecolor = "black", width = 0.5, linewidth = 2)
ax_contri.set_ylabel("Fraction of contribution (-)", fontsize = 15)
ax_contri.tick_params(axis = "x", labelsize = 16.5)
ax_contri.tick_params(axis = "y", labelsize = 12)
ax_contri.text(0.9, 0.9, "(d)", ha = 'left', va = 'bottom', fontsize = 18, transform  = ax_contri.transAxes)

contri.suptitle("Weather regime contribution", fontsize = 20, x = 0, y = 1, ha = 'left', va = 'bottom')
#################################################################################
#legend
# Anticyclonic

mksize = 10  #markersize
hdlen = 3  #handellength
lw = 1.5  #linewidth
mkew = 2  #markeredgewidth


EuBL = mlines.Line2D([], [], color=wr_color_dict['EuBL'], linewidth=lw, label='EuBL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ScBL = mlines.Line2D([], [], color=wr_color_dict['ScBL'], linewidth=lw, label='ScBL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
AR = mlines.Line2D([], [], color=wr_color_dict['AR'], linewidth=lw, label='AR', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
GL = mlines.Line2D([], [], color=wr_color_dict['GL'], linewidth=lw, label='GL', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ScTr = mlines.Line2D([], [], color=wr_color_dict['ScTr'], linewidth=lw, label='ScTr', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
AT = mlines.Line2D([], [], color=wr_color_dict['AT'], linewidth=lw, label='AT', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
ZO = mlines.Line2D([], [], color=wr_color_dict['ZO'], linewidth=lw, label='ZO', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)
no = mlines.Line2D([], [], color=wr_color_dict['no'], linewidth=lw, label='no', linestyle = "-", marker = "x", markersize = mksize, markeredgewidth = mkew)

wr.legend(handles = [EuBL, ScBL, AR, GL, ScTr, AT, ZO, no], ncol = 4, title = "Weather regime", bbox_to_anchor = (0.25, 0.1), frameon = False, handlelength = hdlen, title_fontsize = 18, loc = "center", fontsize = 17)

#---------------------------------
#wr.savefig("/home/lixinh/SynFire_Scripts/WP1/Figures/R_CEE_weather_regime_wo_2001_2002.png", dpi = 600, bbox_inches = "tight")